# Section 4 — Binary Encoding of the Cell Index

This notebook tests the implementation of **Section 4**:
1. **Definition 3 (Cell bits)**: Allocating $b_f = \lceil \log_2 \text{cells}(f) \rceil$ boolean variables $B_f^0, \dots, B_f^{b_f - 1}$ per feature.
2. **Remark 1 (No constraint required & Aliasing)**: Every bit assignment corresponds to exactly one cell without needing mutual-exclusion clauses; surplus bit patterns clamp to the last cell.
3. **Remark 2 (Logarithmic savings)**: Comparing one-hot variable counts ($K$ variables) with binary encoding ($\lceil \log_2 K \rceil$ variables).
4. **Encoding and Decoding**: Bijective mapping on valid cells, plus metadata retrieval (`bit_owner`).

In [6]:
import os
import sys
import numpy as np

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from rfbdd.forest import Tree, Forest
from rfbdd.discretize import Discretization, discretize

def check(condition: bool, message: str) -> None:
    status = "PASS" if condition else "FAIL"
    print(f"[{status}] {message}")
    assert condition, message

print("Setup complete.")

Setup complete.


## 4.1 Sample Forest and Bit Allocation (Definition 3)

> **Definition 3 (Cell bits)**: *For each feature $f$ introduce $b_f = \lceil \log_2 \text{cells}(f) \rceil$ propositional variables $B_f^0, \dots, B_f^{b_f - 1}$.*

We use the test forest from Section 3 where:
- Feature 0 has 4 cells $\implies b_0 = \lceil \log_2 4 \rceil = 2$ bits (`x0_0`, `x0_1`).
- Feature 1 has 2 cells $\implies b_1 = \lceil \log_2 2 \rceil = 1$ bit (`x1_0`).
- Total input bits: $2 + 1 = 3$ bits.

In [7]:
t0 = Tree(
    feature=np.array([0, -1, 1, -1, -1], dtype=np.int64),
    threshold=np.array([5.7, np.nan, 10.0, np.nan, np.nan], dtype=np.float64),
    left=np.array([1, -1, 3, -1, -1], dtype=np.int64),
    right=np.array([2, -1, 4, -1, -1], dtype=np.int64),
    label=np.array([-1, 0, -1, 0, 1], dtype=np.int64),
)
t1 = Tree(
    feature=np.array([0, -1, 0, -1, -1], dtype=np.int64),
    threshold=np.array([2.5, np.nan, 8.1, np.nan, np.nan], dtype=np.float64),
    left=np.array([1, -1, 3, -1, -1], dtype=np.int64),
    right=np.array([2, -1, 4, -1, -1], dtype=np.int64),
    label=np.array([-1, 0, -1, 1, 0], dtype=np.int64),
)
forest = Forest(trees=[t0, t1], n_features=2, n_classes=2, classes=np.array([0, 1]), name="toy_forest")
disc = discretize(forest)

check(disc.n_bits == [2, 1], f"Bit counts per feature: {disc.n_bits} (expected [2, 1])")
check(disc.n_input_bits == 3, f"Total input bits: {disc.n_input_bits} (expected 3)")
check(disc.bit_names == [["x0_0", "x0_1"], ["x1_0"]], f"Bit names: {disc.bit_names}")
check(disc.all_bit_names == ["x0_0", "x0_1", "x1_0"], f"All bit names: {disc.all_bit_names}")

[PASS] Bit counts per feature: [2, 1] (expected [2, 1])
[PASS] Total input bits: 3 (expected 3)
[PASS] Bit names: [['x0_0', 'x0_1'], ['x1_0']]
[PASS] All bit names: ['x0_0', 'x0_1', 'x1_0']


## 4.2 Cell and Sample Encoding

Each integer cell index is converted into its standard binary bit representation:
- Cell `0` $\rightarrow 00_2$ (`False`, `False`)
- Cell `1` $\rightarrow 01_2$ (`True`, `False`)
- Cell `2` $\rightarrow 10_2$ (`False`, `True`)
- Cell `3` $\rightarrow 11_2$ (`True`, `True`)

In [8]:
cases = [
    (np.array([0, 0]), {"x0_0": False, "x0_1": False, "x1_0": False}),
    (np.array([1, 0]), {"x0_0": True, "x0_1": False, "x1_0": False}),
    (np.array([2, 0]), {"x0_0": False, "x0_1": True, "x1_0": False}),
    (np.array([3, 1]), {"x0_0": True, "x0_1": True, "x1_0": True}),
]

for cells, expected in cases:
    encoded = disc.encode_cells(cells)
    check(encoded == expected, f"encode_cells({cells.tolist()}) == {expected}")

# Sample-level encoding directly from continuous coordinates [7.0, 10.1] -> cells [2, 1]
sample = np.array([7.0, 10.1])
sample_encoded = disc.encode_sample(sample)
check(sample_encoded == {"x0_0": False, "x0_1": True, "x1_0": True}, f"encode_sample({sample.tolist()}) correct")

[PASS] encode_cells([0, 0]) == {'x0_0': False, 'x0_1': False, 'x1_0': False}
[PASS] encode_cells([1, 0]) == {'x0_0': True, 'x0_1': False, 'x1_0': False}
[PASS] encode_cells([2, 0]) == {'x0_0': False, 'x0_1': True, 'x1_0': False}
[PASS] encode_cells([3, 1]) == {'x0_0': True, 'x0_1': True, 'x1_0': True}
[PASS] encode_sample([7.0, 10.1]) correct


## 4.3 Decoding and Remark 1 (Clamping of Surplus Bit Patterns)

> **Remark 1 (No constraint required)**: *When the encoded integer exceeds $\text{cells}(f) - 1$, the value is interpreted as the last cell, which already accumulates all the larger values.*

1. For valid cell indices, decoding must be the exact inverse of encoding.
2. For a feature with 3 cells (`n_cells = 3`), 2 bits can represent integers 0, 1, 2, 3. The surplus integer 3 must clamp to the last cell index `2`.

In [9]:
# 1. Invertibility on all valid cell combinations
for c0 in range(disc.cells(0)):
    for c1 in range(disc.cells(1)):
        original = np.array([c0, c1])
        decoded = disc.decode_cells(disc.encode_cells(original))
        check(np.array_equal(original, decoded), f"Reversibility for cells [{c0}, {c1}]")

# 2. Test clamping on a custom discretization with 3 cells (requires 2 bits)
disc_3cells = Discretization(
    thresholds=[np.array([1.0, 2.0])],
    n_cells=[3],
    n_bits=[2],
    bit_names=[["x0_0", "x0_1"]],
    n_features=1,
    taus=[{0: 0, 1: 1}],
)
# Bit assignment for integer 3 (x0_0=True, x0_1=True)
decoded_clamped = disc_3cells.decode_cells({"x0_0": True, "x0_1": True})
check(decoded_clamped[0] == 2, f"Surplus pattern 3 clamped to last cell 2: {decoded_clamped[0]}")

[PASS] Reversibility for cells [0, 0]
[PASS] Reversibility for cells [0, 1]
[PASS] Reversibility for cells [1, 0]
[PASS] Reversibility for cells [1, 1]
[PASS] Reversibility for cells [2, 0]
[PASS] Reversibility for cells [2, 1]
[PASS] Reversibility for cells [3, 0]
[PASS] Reversibility for cells [3, 1]
[PASS] Surplus pattern 3 clamped to last cell 2: 2


## 4.4 Bit Metadata and Variable Savings

1. **Bit ownership**: Rapidly querying which feature and bit position a variable belongs to.
2. **Remark 2 (Variable savings)**: Comparing standard one-hot vs binary encoding variable count.

In [10]:
owner = disc.bit_owner()
expected_owner = {"x0_0": (0, 0), "x0_1": (0, 1), "x1_0": (1, 0)}
check(owner == expected_owner, f"bit_owner mapping: {owner}")

# Remark 2 illustration
sample_cell_counts = [2, 4, 8, 16, 32, 64, 128]
for k in sample_cell_counts:
    one_hot_vars = k
    binary_vars = int(np.ceil(np.log2(k)))
    savings = (1 - binary_vars / one_hot_vars) * 100
    print(f"Cells: {k:3d} | One-hot vars: {one_hot_vars:3d} | Binary bits: {binary_vars:2d} | Reduction: {savings:5.1f}%")

[PASS] bit_owner mapping: {'x0_0': (0, 0), 'x0_1': (0, 1), 'x1_0': (1, 0)}
Cells:   2 | One-hot vars:   2 | Binary bits:  1 | Reduction:  50.0%
Cells:   4 | One-hot vars:   4 | Binary bits:  2 | Reduction:  50.0%
Cells:   8 | One-hot vars:   8 | Binary bits:  3 | Reduction:  62.5%
Cells:  16 | One-hot vars:  16 | Binary bits:  4 | Reduction:  75.0%
Cells:  32 | One-hot vars:  32 | Binary bits:  5 | Reduction:  84.4%
Cells:  64 | One-hot vars:  64 | Binary bits:  6 | Reduction:  90.6%
Cells: 128 | One-hot vars: 128 | Binary bits:  7 | Reduction:  94.5%
